In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "delta-spark==3.1.0", "python-dotenv", "pymongo"], check=True)
print("Done")

Done


In [2]:
import os, uuid, json, logging
from datetime import datetime, date
from dotenv import load_dotenv, find_dotenv
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

load_dotenv(find_dotenv(), override=True)

CONFIG = {
    "minio_endpoint":    os.getenv("MINIO_ENDPOINT",    "http://minio:9000"),
    "minio_access_key":  os.getenv("MINIO_ACCESS_KEY",  ""),
    "minio_secret_key":  os.getenv("MINIO_SECRET_KEY",  ""),
    "bronze_bucket":     os.getenv("BRONZE_BUCKET",     "bronze"),
    "silver_bucket":     os.getenv("SILVER_BUCKET",     "silver"),
    "gold_bucket":       os.getenv("GOLD_BUCKET",       "gold"),
    "quarantine_bucket": os.getenv("QUARANTINE_BUCKET", "quarantine"),
    "mongodb_uri":       os.getenv("MONGODB_URI",       ""),
    "mongodb_db":        os.getenv("MONGODB_DB",        "atlas_pipeline"),
    "postgres_uri":      os.getenv("POSTGRES_URI",      ""),
    "kafka_broker":      "kafka:29092",
    "kafka_topics":      "bank.pko.transactions,"
                         "bank.mbank.transactions,"
                         "bank.ing.transactions",
    "pipeline_name":     "atlas-stream-bronze",
    "processing_date":   str(date.today()),
    "trigger_interval":  "10 seconds",
    "checkpoint_path":   "s3a://bronze/checkpoints/streaming/",
}

RUN_ID = str(uuid.uuid4())
print(f"Pipeline:   {CONFIG['pipeline_name']}")
print(f"Run ID:     {RUN_ID}")
print(f"Kafka:      {CONFIG['kafka_broker']}")
print(f"Topics:     {CONFIG['kafka_topics']}")
print("Configuration loaded")

Pipeline:   atlas-stream-bronze
Run ID:     f51e81d8-f7b7-49c9-a10f-18231282beec
Kafka:      kafka:29092
Topics:     bank.pko.transactions,bank.mbank.transactions,bank.ing.transactions
Configuration loaded


In [3]:
import os
from pyspark.sql import SparkSession

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "io.delta:delta-spark_2.12:3.1.0,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.367,"
    "org.apache.spark:spark-token-provider-kafka-0-10_2.12:3.5.0,"
    "org.apache.commons:commons-pool2:2.11.1 "
    "pyspark-shell"
)

try:
    SparkSession.getActiveSession().stop()
    import time; time.sleep(2)
    print("Stopped existing session")
except:
    pass

spark = (
    SparkSession.builder
    .appName("Atlas — Stream Bronze Ingestion")
    .master("local[2]")
    .config("spark.sql.extensions",          "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog","org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions",   "4")
    .config("spark.hadoop.fs.s3a.endpoint",   CONFIG["minio_endpoint"])
    .config("spark.hadoop.fs.s3a.access.key", CONFIG["minio_access_key"])
    .config("spark.hadoop.fs.s3a.secret.key", CONFIG["minio_secret_key"])
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .config("spark.hadoop.fs.s3a.connection.maximum", "100")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version:  {spark.version}")
print(f"Spark master:   {spark.sparkContext.master}")
print(f"Spark UI:       http://localhost:4040")
print("Spark session ready")

Spark version:  3.5.0
Spark master:   local[2]
Spark UI:       http://localhost:4040
Spark session ready


In [4]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F

transaction_schema = StructType([
    StructField("transaction_id",    StringType(), nullable=True),
    StructField("customer_id",       StringType(), nullable=True),
    StructField("account_id",        StringType(), nullable=True),
    StructField("merchant_id",       StringType(), nullable=True),
    StructField("merchant_name",     StringType(), nullable=True),
    StructField("merchant_category", StringType(), nullable=True),
    StructField("amount",            StringType(), nullable=True),
    StructField("currency",          StringType(), nullable=True),
    StructField("transaction_type",  StringType(), nullable=True),
    StructField("status",            StringType(), nullable=True),
    StructField("timestamp",         StringType(), nullable=True),
    StructField("bank_code",         StringType(), nullable=True),
    StructField("country",           StringType(), nullable=True),
    StructField("device_type",       StringType(), nullable=True),
    StructField("ip_address",        StringType(), nullable=True),
])

print(f"Schema defined — {len(transaction_schema.fields)} fields")

kafka_raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", CONFIG["kafka_broker"])
    .option("subscribe",               CONFIG["kafka_topics"])
    .option("startingOffsets",         "latest")
    .option("failOnDataLoss",          "false")
    .option("maxOffsetsPerTrigger",    "100")
    .load()
)

parsed_stream = (
    kafka_raw_stream
    .select(
        F.col("value").cast("string").alias("json_string"),
        F.col("topic").alias("kafka_topic"),
        F.col("partition").alias("kafka_partition"),
        F.col("offset").alias("kafka_offset"),
        F.col("timestamp").alias("kafka_timestamp"),
    )
    .withColumn("data", F.from_json(F.col("json_string"), transaction_schema))
    .select("data.*", "kafka_topic", "kafka_partition", "kafka_offset", "kafka_timestamp")
)

print("Kafka stream connected")
print("Parsed stream schema:")
parsed_stream.printSchema()

Schema defined — 15 fields
Kafka stream connected
Parsed stream schema:
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- status: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- bank_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- device_type: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)



In [5]:
from pyspark.sql import functions as F

bronze_stream = (
    parsed_stream
    .withColumn("ingested_at",     F.current_timestamp())
    .withColumn("pipeline_run_id", F.lit(RUN_ID))
    .withColumn("processing_date", F.current_date().cast("string"))
)

BRONZE_DELTA_PATH = f"s3a://{CONFIG['bronze_bucket']}/delta/transactions/"
CHECKPOINT_PATH   = f"s3a://{CONFIG['bronze_bucket']}/checkpoints/bronze-stream/"

print(f"Bronze path:    {BRONZE_DELTA_PATH}")
print(f"Checkpoint:     {CHECKPOINT_PATH}")
print("Starting stream...")

bronze_query = (
    bronze_stream
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .partitionBy("processing_date", "bank_code")
    .trigger(processingTime="10 seconds")
    .start(BRONZE_DELTA_PATH)
)

print(f"Stream ID:     {bronze_query.id}")
print(f"Stream active: {bronze_query.isActive}")

Bronze path:    s3a://bronze/delta/transactions/
Checkpoint:     s3a://bronze/checkpoints/bronze-stream/
Starting stream...
Stream ID:     5e9a1240-087c-430c-954c-12666dc7cabf
Stream active: True


In [6]:
# ─────────────────────────────────────────────────────────────
# Cell 6 — Monitor the Bronze streaming pipeline
#
# The stream is running in the background
# This cell checks progress without stopping it
#
# We check three things:
#   1. Is the stream still active?
#   2. How many records processed per batch?
#   3. How many records landed in Bronze Delta Lake?
# ─────────────────────────────────────────────────────────────

import time
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# STEP 1 — Stream health check
# ─────────────────────────────────────────────────────────────
print("=" * 55)
print("STREAM STATUS")
print("=" * 55)
print(f"Active:     {bronze_query.isActive}")
print(f"Query ID:   {bronze_query.id}")
print(f"Exception:  {bronze_query.exception()}")

# ─────────────────────────────────────────────────────────────
# STEP 2 — Wait for micro-batches then check progress
# Wait 30 seconds — allows 2 to 3 micro-batches to complete
# Each micro-batch runs every 10 seconds
# ─────────────────────────────────────────────────────────────
print()
print("Waiting 30 seconds for micro-batches...")
time.sleep(30)

progress = bronze_query.lastProgress

if progress:
    print()
    print("=" * 55)
    print("LATEST MICRO-BATCH")
    print("=" * 55)
    print(f"Batch ID:        {progress['batchId']}")
    print(f"Timestamp:       {progress['timestamp']}")
    print(f"Input rows:      {progress['numInputRows']}")
    print(f"Rows per second: {progress['processedRowsPerSecond']:.2f}")
    print()
    print("Sources:")
    for source in progress.get("sources", []):
        print(f"  {source.get('description', '')[:60]}")
        print(f"  Rows: {source.get('numInputRows', 0)}")
else:
    print("No progress yet — rerun this cell in 10 seconds")

# ─────────────────────────────────────────────────────────────
# STEP 3 — Read Bronze Delta Lake and count records
#
# Reading Delta as batch is separate from streaming write
# Does not interfere with the running stream
# ─────────────────────────────────────────────────────────────
print()
print("=" * 55)
print("BRONZE DELTA LAKE")
print("=" * 55)

BRONZE_DELTA_PATH = f"s3a://{CONFIG['bronze_bucket']}/delta/transactions/"

try:
    bronze_df = spark.read.format("delta").load(BRONZE_DELTA_PATH)
    total     = bronze_df.count()

    print(f"Total records: {total}")
    print()

    print("Records per bank:")
    bronze_df.groupBy("bank_code") \
        .agg(F.count("*").alias("count")) \
        .orderBy("bank_code") \
        .show()

    print("Records per status:")
    bronze_df.groupBy("status") \
        .agg(F.count("*").alias("count")) \
        .orderBy("status") \
        .show()

    print("Latest 5 records:")
    bronze_df.orderBy(F.col("ingested_at").desc()) \
        .select(
            "transaction_id",
            "bank_code",
            "amount",
            "currency",
            "status",
            "ingested_at"
        ).show(5, truncate=False)

except Exception as e:
    print(f"Delta table not ready yet: {e}")
    print("Wait 10 seconds and rerun this cell")

STREAM STATUS
Active:     True
Query ID:   5e9a1240-087c-430c-954c-12666dc7cabf
Exception:  None

Waiting 30 seconds for micro-batches...

LATEST MICRO-BATCH
Batch ID:        18
Timestamp:       2026-04-08T08:35:20.001Z
Input rows:      15
Rows per second: 2.51

Sources:
  KafkaV2[Subscribe[bank.pko.transactions, bank.mbank.transact
  Rows: 15

BRONZE DELTA LAKE
Total records: 268

Records per bank:
+---------+-----+
|bank_code|count|
+---------+-----+
|      ING|  119|
|    MBANK|   59|
|      PKO|   90|
+---------+-----+

Records per status:
+---------+-----+
|   status|count|
+---------+-----+
|completed|  269|
|  pending|   14|
+---------+-----+

Latest 5 records:
+----------------------+---------+------+--------+---------+-----------------------+
|transaction_id        |bank_code|amount|currency|status   |ingested_at            |
+----------------------+---------+------+--------+---------+-----------------------+
|TXN-ING-D5D7AB14FB1B  |ING      |158.41|EUR     |completed|2026-04-